# KF M1+M9 EnbPI

Giordano M1 and M9 conditional-mean equations combined additively (retaining the project's coupled volatility and observation noise) → causal Kalman decomposition → B low-frequency rolling-refit ARIMA models + B high-frequency point ANNs trained with MSE → official-style nested OOB mean aggregation and EnbPI calibration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kf_enbpi import (
    EnbPIConfig, simulate_and_run, monte_carlo_summary,
    plot_result, plot_monte_carlo_summary, plot_monte_carlo_forecast_diagnostics,
)

pd.set_option('display.precision', 4)

## Experiment configuration
`batch_size=1` means the response is revealed after each forecast. For larger batches, forecasts are recursive inside the batch and truths update the residual pool only after the whole batch. The low and high panels use separate online EnbPI residual pools calibrated against the causal KF component references.

In [ ]:
config = EnbPIConfig(
    window_size=15,
    alpha=0.05,
    n_bootstrap=30,
    block_length=None,  # defaults to round(sqrt(number of training windows))
    batch_size=1,
    beta_grid_size=101,
    arima_order=None,  # BIC selection on KF low train; rolling re-fit at every forecast step
    arima_max_p=4,
    arima_max_q=4,
    ann_hidden_layers=(32, 16),
    ann_max_iter=500,
    ann_alpha=1e-4,
    ann_learning_rate_init=1e-3,
    ann_early_stopping=True,
    ann_tol=1e-3,
    random_state=1234,
)
train_size = 650
horizon = 50

## One reproducible run

In [ ]:
result = simulate_and_run(
    'm1m9', train_size=train_size, horizon=horizon, config=config, data_seed=2026
)
print(f'Actual result: train_size={result.train_size}, forecast_start={result.test_times[0]}, total_length={len(result.observed)}')
final_summary, component_summary = result.summary_tables()
print(f'Data seed={result.data_seed}, ensemble seed={result.config.random_state}')
display(final_summary)
display(component_summary)
display(result.frame().head())
plot_result(result);

## Diagnostics
Every training target must have at least one OOB model. The residual plots help assess the short-term i.i.d./mixing assumptions used by the EnbPI theory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(result.initial_oob_residuals, bins=25, edgecolor='white')
axes[0].set_title('Initial OOB residual distribution')
axes[1].plot(result.initial_oob_residuals, linewidth=1)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('OOB residual sequence')
fig.tight_layout();
print('OOB model counts:', result.oob_counts.min(), result.oob_counts.mean(), result.oob_counts.max())

## Monte Carlo summary

In [ ]:
runs, summary, mc_results = monte_carlo_summary(
    'm1m9', n_runs=20, train_size=train_size, horizon=horizon, config=config, seed=2026
)
display(summary)
display(runs)
plot_monte_carlo_forecast_diagnostics(
    runs, mc_results, model_name='m1m9', alpha=config.alpha
);